In [ ]:
import os 

os.chdir("..")

In [ ]:
from jp_imports import JPTrade
from jp_qcew import CleanQCEW
import polars as pl
import pymc as pm
import pandas as pd
from prprod_repl import ProdRepl
import arviz_base as azb
import arviz_plots as azp
azp.style.use("arviz-variat")


jt = JPTrade()
pr = ProdRepl()
cq = CleanQCEW(saving_dir="data/")

In [ ]:
gdf = pr.county_geom()[["name", "geoid"]]
gdf = pl.DataFrame(gdf)
df2 = gdf.with_columns(
        pl.col("name")
            .str.normalize("NFD")
            .str.replace_all(r"[\u0300-\u036f]", "")
            .str.to_lowercase()
            .str.replace_all(" ", "_")
        )
df2 = df2.to_pandas()
df2

In [ ]:
df = pr.data_qcew()
df = df.with_columns(
    name=(pl.col("phys_addr_city")
            .str.normalize("NFD")
            .str.replace_all(r"[\u0300-\u036f]", "")
            .str.to_lowercase()
            .str.replace_all("  ", "_")
            .str.replace_all(" ", "_")
        )
)
df = df.to_pandas()
df

In [ ]:
import pandas as pd
from thefuzz import process, fuzz

# 1. Define a function to find the best match
def get_best_match(name, choices, threshold=80):
    # process.extractOne returns (match, score, index)
    match, score = process.extractOne(name, choices, scorer=fuzz.token_sort_ratio)
    
    # Only return the match if it's "good enough"
    return match if score >= threshold else None

# 2. Get the list of 'correct' names from df2
correct_names = df2['name'].tolist()

# 3. Create a new column in df that maps to the closest correct name
df['matched_name'] = df['name'].apply(lambda x: get_best_match(x, correct_names))

# 4. Merge df with df2 based on the new matched column
result = pd.merge(df, df2, left_on='matched_name', right_on='name', suffixes=('_original', '_cleaned'))

In [ ]:
df_imports = jt.process_int_jp(level="naics", time_frame="qrt")
df_imports = df_imports.with_columns(
    sector_code=pl.col("naics").str.slice(0,2)
).rename({"qrt":"qtr"})
df_imports = df_imports.group_by(["year", "qtr", "sector_code"]).agg(pl.col("imports", "imports_qty", "exports","exports_qty","net_exports","net_qty").sum())


In [ ]:
data = pl.DataFrame(result)
data = data.with_columns(name="name_cleaned")
data = data.group_by(["year", "month", "sector_code", "qtr", "name"]).agg(pl.col("employment").sum())
data.join(df_imports,on=["year", "qtr","sector_code"], how="inner", validate="m:1")